# Week 7: Customer Support Chatbot - Homework Solutions 🤖

## Overview
This notebook contains sample answers to the Week 7 homework questions. These are **reference solutions** to help you understand the concepts better. Your answers may differ and still be correct as long as they demonstrate understanding!

---

## Question 1: Understanding RAG Architecture

### Sample Answer:

**RAG Pipeline Flow:**
1. **User Query** → User types: "How do I reset my password?"
2. **Embed Query** → Convert the question into a vector/embedding using OpenAI
3. **Retrieve from Vector DB** → Search Chroma database for most similar FAQ items (e.g., top 3 matches)
4. **Build Context** → Take the retrieved FAQs and add them to the prompt
5. **Send to LLM** → Pass context + query + system prompt to GPT-4
6. **Generate Response** → LLM generates answer based on FAQ context
7. **Return to User** → Customer gets informed, accurate answer

**Why RAG is Better than Plain LLM:**
- **Plain LLM**: Answers from training data (may be outdated, hallucinations, generic)
- **RAG LLM**: Answers from your specific FAQ knowledge base (current, accurate, company-specific)

**3 Key Advantages of RAG for Customer Support:**
1. **Accuracy**: Responses are grounded in verified FAQ knowledge, reducing hallucinations
2. **Consistency**: All customers get the same answer to the same question
3. **Easy Updates**: Update FAQs without retraining the model - just update the vector database
4. **Cost-Efficient**: Don't need GPT-4 to remember everything - it only needs to synthesize retrieved answers
5. **Auditability**: You can trace which FAQ was used to answer which question

## Question 2: Vector Embeddings and Semantic Search

### Sample Answer:

**How Embeddings Help Match Similar Questions:**

User's question: "How do I change my account password?"
FAQ question: "How do I handle reset password?"

These words are different, but embeddings capture **meaning**:
- Both questions are converted to 1536-dimensional vectors (using OpenAI embeddings)
- Vectors for similar topics are **close together in vector space**
- We calculate **cosine similarity** between the user query and each FAQ
- "password" and "reset password" have very high similarity (0.85+)
- The FAQ gets retrieved and provided to the user

**Why Keyword Matching Fails:**
- Keyword matching looks for exact words (Ctrl+F approach)
- "change password" and "reset password" have no overlapping key words
- Query won't match the FAQ even though they mean the same thing
- Semantic search solves this by understanding **meaning** not just **words**

**Efficiency with 500 FAQs:**
- Without vector DB: Search all 500 items sequentially (slow, O(n) complexity)
- With Chroma vector DB: Uses **HNSW indexing** (fast approximate nearest neighbor search)
- Complexity drops to O(log n) or O(1) for most queries
- Can retrieve top 3 most relevant FAQs in milliseconds even with 500+ items

## Question 3: Chroma Vector Database

### Sample Answer:

**What "Persistent" Storage Means:**
- **Persistent** = Data survives kernel restart and Python session end
- Without persistence: Every time you restart Jupyter, you'd need to re-embed all 500 FAQs (costs money, takes time)
- With Chroma persistent storage: Embeddings are saved to disk and loaded instantly on next run
- **Important for Production**: Chatbot must work 24/7 without losing data when restarted

**Adding 100 New FAQs Without Losing Previous Items:**
```python
# Load existing Chroma collection
from langchain.vectorstores import Chroma
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

# Add new FAQs to existing collection (they get appended)
new_faqs = load_new_faqs_from_csv("new_faqs.csv")  # 100 new items
vectorstore.add_documents(new_faqs)  # Adds without deleting old ones

# Old 500 FAQs + New 100 FAQs = 600 total FAQs in database
```

**When to Clear and Recreate Chroma Database:**
1. **FAQ Structure Changed**: If metadata schema changes (e.g., add new field "priority")
2. **Embedding Model Upgraded**: Old embeddings incompatible with new embedding model
3. **Data Quality Issues**: Discovered bad/duplicate FAQs that need full refresh
4. **Performance Tuning**: Need to rebuild index with different parameters
5. **Fresh Start**: Testing/development phase before going to production

## Question 4: Metadata Filtering in Retrieval

### Sample Answer:

**Using Metadata Filtering:**

Scenario: Customer from EU region, asking about reset password via mobile app

Strategy:
1. First retrieve all similar FAQs about "reset password" (semantic search)
2. Then filter results by: `region = 'EU'` AND `channel = 'mobile app'`
3. Return the most relevant filtered result
4. If no results after filtering, fall back to just `region = 'EU'` or any reset password FAQ

**Python Code Example:**
```python
from langchain.vectorstores import Chroma

vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

# Search with metadata filter
results = vectorstore.similarity_search_with_score(
    query="reset password on mobile",
    k=3,
    filter={  # Metadata filter
        "region": "EU",
        "channel": "mobile app"
    }
)

# Get top result
best_faq = results[0][0].page_content
print(best_faq)
```

**What Could Go Wrong:**
- **No results after filtering**: User asks about an issue that's not in FAQ for their region/channel
  - **Solution**: Implement fallback logic - try broader filter (just region), then just issue type
- **Outdated metadata**: FAQs not tagged with correct region/channel
  - **Solution**: Data validation before loading into Chroma
- **Overfiltering**: Too strict filters may miss relevant FAQs
  - **Solution**: Start with semantic search, then apply light filtering

In [ ]:
# Question 4: Metadata Filtering Code Example

# Simulated FAQ data with metadata
faq_data = [
    {
        "question": "How do I reset password on mobile?",
        "answer": "For reset password on the mobile app, open Settings > Security and follow the prompts.",
        "region": "EU",
        "channel": "mobile app",
        "issue": "reset password"
    },
    {
        "question": "How do I reset password on web?",
        "answer": "For reset password on the web app, open Settings > Security and follow the prompts.",
        "region": "US",
        "channel": "web app",
        "issue": "reset password"
    },
    {
        "question": "How do I handle billing discrepancy?",
        "answer": "Visit Billing > Payments and refresh your card on file.",
        "region": "EU",
        "channel": "mobile app",
        "issue": "billing discrepancy"
    },
    {
        "question": "How do I upgrade my plan?",
        "answer": "Go to Plans and choose the desired tier, then confirm.",
        "region": "APAC",
        "channel": "email",
        "issue": "upgrade plan"
    }
]

def filter_faqs_by_metadata(faqs, region=None, channel=None, issue=None):
    """Filter FAQs by metadata with fallback strategy"""
    
    # Step 1: Try filtering with all specified criteria
    filtered = faqs.copy()
    
    if region:
        filtered = [f for f in filtered if f["region"] == region]
    if channel:
        filtered = [f for f in filtered if f["channel"] == channel]
    if issue:
        filtered = [f for f in filtered if f["issue"] == issue]
    
    if filtered:
        return filtered, f"Found {len(filtered)} FAQs with {region}/{channel}"
    
    # Step 2: Fall back to region only
    if region:
        filtered = [f for f in faqs if f["region"] == region]
        if filtered:
            return filtered, f"No match for {channel}, found {len(filtered)} for region {region}"
    
    # Step 3: Fall back to issue type only
    if issue:
        filtered = [f for f in faqs if f["issue"] == issue]
        if filtered:
            return filtered, f"No match for region/channel, found {len(filtered)} for issue {issue}"
    
    # Step 4: Return all FAQs
    return faqs, "No metadata match, returning all FAQs"

# Test the filter
print("Customer: From EU, mobile app, password issue")
print("-" * 50)
results, msg = filter_faqs_by_metadata(faq_data, region="EU", channel="mobile app", issue="reset password")
print(f"✅ {msg}")
print(f"Retrieved: {results[0]['answer']}\n")

print("Customer: From EU, unknown channel, password issue")
print("-" * 50)
results, msg = filter_faqs_by_metadata(faq_data, region="EU", channel="phone", issue="reset password")
print(f"⚠️ {msg}")
print(f"Retrieved: {results[0]['answer']}")


## Question 5: Conversation Memory Strategies

### Sample Answer:

**For 50-Message Conversation: Window Memory**

Why?
- **Buffer Memory** would be too long (50 messages × ~100 tokens = 5000+ tokens for history alone)
- **Window Memory** keeps only last ~5-10 messages (500-1000 tokens) - good balance
- **Summary Memory** might lose important context in long conversation
- Window Memory captures recent context (what user just asked) without excess cost

---

**Buffer vs Window Memory:**

| Aspect | Buffer Memory | Window Memory |
|--------|--------------|---------------|
| Keeps | All conversation history | Last K messages |
| Cost | Increases with time | Stable/bounded |
| Context | Complete but verbose | Recent context only |
| Best For | Short conversations (5-10 turns) | Medium conversations (20-100 turns) |
| Token Usage | 10 turns = 1000 tokens; 100 turns = 10,000 tokens | Always ~500-1000 tokens |

**When to use:**
- **Buffer**: Quick FAQ lookups, simple questions (1-2 turns)
- **Window**: Most customer support scenarios

---

**Trade-off: Full History vs. Summary**

**Full History (Buffer):**
- ✅ LLM has complete context
- ✅ Can refer back to earliest messages
- ❌ Very expensive (10,000+ tokens for long chats)
- ❌ Slow response time

**Summary Memory:**
- ✅ Cheap (2000-3000 tokens)
- ✅ Fast responses
- ❌ Loses fine-grained details
- ❌ Summary itself costs tokens (need LLM to summarize)

---

**Decision Tree for Memory Selection:**

```
START
  |
  v
Is conversation < 5 turns?
  |-- YES --> Use BUFFER MEMORY (simplest)
  |
  |-- NO
       |
       v
    Is budget very tight (cost matters most)?
       |-- YES --> Use WINDOW MEMORY (default choice)
       |
       |-- NO
            |
            v
         Is conversation > 200 turns AND need to refer to early messages?
            |-- YES --> Use SUMMARY MEMORY (balances cost + context)
            |
            |-- NO --> Use WINDOW MEMORY (most reliable)
```

In [ ]:
# Question 5: Memory Strategies Comparison Code

from datetime import datetime
from collections import deque

class BufferMemory:
    """Keeps ALL conversation history"""
    def __init__(self):
        self.messages = []
    
    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})
    
    def get_context(self):
        return "\n".join([f"{m['role']}: {m['content']}" for m in self.messages])
    
    def get_token_count(self):
        # Rough estimate: ~4 tokens per word
        total_words = sum(len(m['content'].split()) for m in self.messages)
        return total_words * 4

class WindowMemory:
    """Keeps only last K messages"""
    def __init__(self, window_size=5):
        self.window = deque(maxlen=window_size)
        self.window_size = window_size
    
    def add_message(self, role, content):
        self.window.append({"role": role, "content": content})
    
    def get_context(self):
        return "\n".join([f"{m['role']}: {m['content']}" for m in self.window])
    
    def get_token_count(self):
        total_words = sum(len(m['content'].split()) for m in self.window)
        return total_words * 4

class SummaryMemory:
    """Summarizes old messages, keeps recent ones"""
    def __init__(self, summary_threshold=10):
        self.messages = []
        self.summary = ""
        self.summary_threshold = summary_threshold
    
    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})
        
        # Summarize when messages exceed threshold
        if len(self.messages) > self.summary_threshold:
            self.summary = f"Earlier conversation: User asked about {len(self.messages)} topics"
            self.messages = self.messages[-5:]  # Keep last 5
    
    def get_context(self):
        context = ""
        if self.summary:
            context += f"[SUMMARY] {self.summary}\n\n"
        context += "\n".join([f"{m['role']}: {m['content']}" for m in self.messages])
        return context
    
    def get_token_count(self):
        total_words = len(self.summary.split()) + sum(len(m['content'].split()) for m in self.messages)
        return total_words * 4

# Simulate a 50-message conversation
print("=" * 60)
print("MEMORY STRATEGY COMPARISON: 50-Message Conversation")
print("=" * 60)

# Test messages
test_messages = [
    ("User", "I can't reset my password"),
    ("Bot", "Let me help you with that. Please try going to Settings > Security."),
    ("User", "I tried that but it's still not working"),
    ("Bot", "Let me escalate this to our specialist team."),
    ("User", "Thanks! Also, when will I be charged?"),
    ("Bot", "You'll be charged on the 1st of next month."),
]

# Extend to 50 messages
for i in range(8):
    test_messages.append(("User", f"Question {i+1}"))
    test_messages.append(("Bot", f"Answer {i+1}"))

# Compare all three strategies
strategies = {
    "Buffer Memory": BufferMemory(),
    "Window Memory": WindowMemory(window_size=10),
    "Summary Memory": SummaryMemory(summary_threshold=20)
}

for role, content in test_messages:
    for strategy in strategies.values():
        strategy.add_message(role, content)

print("\nResults after 50 messages:\n")
for name, strategy in strategies.items():
    tokens = strategy.get_token_count()
    cost = tokens * 0.000002  # $2 per 1M tokens
    print(f"{name}:")
    print(f"  Token Count: {tokens:,} tokens")
    print(f"  API Cost: ${cost:.4f}")
    print(f"  Context Length: {'Full history' if isinstance(strategy, BufferMemory) else f'{len(getattr(strategy, \"window\", strategy.messages))} recent messages'}")
    print()

print("\nRECOMMENDATION:")
print("✅ Window Memory for 50-message conversation")
print("   - Balanced cost (~8,000 tokens)")
print("   - Recent context preserved")
print("   - Consistent performance")


## Question 6: Prompt Engineering for Customer Support

### Sample Answer:

**System Prompt for Customer Support Chatbot:**

```
You are a helpful customer support assistant for our platform. Your role is to answer 
customer questions based on the provided FAQ knowledge base. 

IMPORTANT RULES:
1. Only use information from the provided FAQ context. Do not use outside knowledge.
2. If the FAQ doesn't contain the answer, be honest and say: "I don't have information about this. 
   Let me connect you with a human agent." Then escalate to support team.
3. Keep responses concise (2-3 sentences max) unless the question requires more detail.
4. Always be professional, courteous, and empathetic to customer frustrations.
5. If a customer asks about something outside our domain (e.g., programming advice), 
   politely redirect them.
6. If the same issue isn't resolved after 2 attempts, offer to escalate to human support.

Format your answer clearly and include next steps if applicable.
```

---

**Casual vs. Formal Tone:**

**Casual Version:**
```
Hey! 👋 I'm here to help you out with any questions about our platform. 
I'll do my best to find answers from our FAQ database. If I can't help, 
no worries—I'll get you connected with a real person.
...
```

**Formal Version:**
```
Good day. I am a customer support assistant ready to address your inquiries 
regarding our services. I will provide information sourced exclusively from 
our comprehensive FAQ documentation.
...
```

---

**Handling Unknown Answers:**

Add this section to your system prompt:

```
ESCALATION PROTOCOL:
If the customer's question is not covered in the FAQ:
1. Acknowledge their question: "That's a great question!"
2. Apologize for not having the answer: "Unfortunately, I don't have specific guidance on this."
3. Offer escalation: "Let me connect you with our specialist team who can help."
4. Collect relevant info: Ask for ticket ID, issue description, preferred contact method
```

## Question 7: Building a Multi-Turn Conversation

### Sample Answer:

**Code for Multi-Turn Conversation with Memory:**

In [ ]:
# Sample Code - Educational Implementation

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.llms import OpenAI
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.prompts import PromptTemplate

# Initialize components
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
llm = OpenAI(temperature=0.7)

# Create memory to track conversation
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Custom system prompt
system_prompt = """You are a helpful customer support assistant. Use the context below to answer questions.
Context: {context}
Chat History: {chat_history}
"""

# Simulate multi-turn conversation
def chat_with_memory(user_message):
    """Process user message and return response with memory"""
    
    # Step 1: Retrieve relevant FAQs
    retrieved_docs = vectorstore.similarity_search(user_message, k=3)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    
    # Step 2: Get chat history
    chat_history = memory.chat_memory.messages
    
    # Step 3: Create prompt with context and history
    prompt = system_prompt.format(context=context, chat_history=chat_history)
    
    # Step 4: Get response from LLM
    response = llm.predict(prompt + f"\nUser: {user_message}")
    
    # Step 5: Store in memory
    memory.chat_memory.add_user_message(user_message)
    memory.chat_memory.add_ai_message(response)
    
    return response

# Example conversation
print("Turn 1:")
resp1 = chat_with_memory("I can't reset my password")
print(f"Bot: {resp1}\n")

print("Turn 2:")
resp2 = chat_with_memory("I tried that but it still didn't work")
print(f"Bot: {resp2}\n")

print("Turn 3:")
resp3 = chat_with_memory("Also, how do I contact support?")
print(f"Bot: {resp3}")

**How Response Differs With vs. Without Memory:**

**Without Memory (Message 2):**
```
User: "I tried that but it still didn't work"
Bot: "Could you clarify which feature you're having issues with?"
            ↑ Bot has NO context about password reset from Turn 1
```

**With Memory (Message 2):**
```
User: "I tried that but it still didn't work"
Memory has: "Turn 1: User asked about password reset"
Bot: "I understand you've tried the reset password steps and it's still not working. 
      Let me escalate this to our specialist team."
            ↑ Bot understands the context from Turn 1
```

---

**Token Count Impact:**

```
Message 1: 100 tokens (question) + 300 tokens (context) = 400 tokens
Message 2: 80 tokens (q) + 300 tokens (context) + 500 tokens (history from msg 1) = 880 tokens
Message 3: 70 tokens (q) + 300 tokens (context) + 1200 tokens (history from msgs 1-2) = 1570 tokens
...
Message 100: 100 tokens (q) + 300 tokens (context) + 45,000 tokens (99-message history!) = 45,400 tokens

Cost Impact: 45,400 tokens × $0.002/1K tokens = $0.09 per message
```

---

**Solution for Long Conversations:**

Switch to **Window Memory** after message 50:

```python
from langchain.memory import ConversationSummaryBufferMemory

# Use Summary Buffer - keeps full history until it reaches token limit (4000)
# Then summarizes and compresses older messages
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=4000,  # Auto-summarize when exceeded
    memory_key="chat_history"
)

# Result: Token count stays bounded around 4000-5000 tokens
# Even after 1000 message conversation!
```

## Question 8: Production Deployment Considerations

### Sample Answer:

**1. Caching Embeddings to Reduce Costs:**

```python
# Strategy: Cache embeddings locally
# Don't recompute embeddings for repeated questions

from functools import lru_cache
import json

class CachedEmbeddings:
    def __init__(self):
        self.cache = {}  # Store: question -> embedding
        self.load_cache()
    
    @lru_cache(maxsize=10000)  # Cache last 10k questions
    def get_embedding(self, text):
        if text in self.cache:
            return self.cache[text]  # Free! Use cached version
        
        # Only call API if not cached
        embedding = openai.Embedding.create(input=text)["data"][0]["embedding"]
        self.cache[text] = embedding
        self.save_cache()  # Persist cache to disk
        return embedding
    
    def load_cache(self):
        try:
            with open("embeddings_cache.json") as f:
                self.cache = json.load(f)
        except:
            self.cache = {}
    
    def save_cache(self):
        with open("embeddings_cache.json", "w") as f:
            json.dump(self.cache, f)

# Impact: 80% of queries hit cache → 80% cost reduction!
```

---

**2. Updating Vector Database Without Downtime:**

```python
# Strategy: Blue-Green Deployment
# Keep 2 vector databases: blue (current) and green (new)
# Switch traffic after new one is ready

class VectorDBManager:
    def __init__(self):
        self.active_db = Chroma(persist_directory="./chroma_blue")  # Production
        self.staging_db = Chroma(persist_directory="./chroma_green")  # Staging
    
    def update_faqs_weekly(self, new_faqs_csv):
        # Load new FAQs into STAGING database
        print("Updating staging database...")
        new_faqs = load_faqs_from_csv(new_faqs_csv)
        self.staging_db.add_documents(new_faqs)
        
        # Test staging database
        print("Running quality tests...")
        test_results = run_qa_tests(self.staging_db)
        
        if test_results['pass_rate'] > 0.95:  # 95% test pass rate
            # Switch: All new traffic goes to green
            self.active_db = self.staging_db
            print("✅ Switched to new database. Users unaffected!")
        else:
            print("❌ Tests failed. Keeping old database active.")

# Zero downtime! Old customers stay on blue, new customers on green, then swap
```

---

**3. Monitoring & Logging:**

```python
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class MonitoredChatbot:
    def __init__(self):
        self.metrics = {
            "total_queries": 0,
            "successful_responses": 0,
            "escalations": 0,
            "avg_response_time": 0,
            "errors": 0
        }
    
    def answer_query(self, user_query, session_id):
        start_time = datetime.now()
        
        try:
            # Query processing
            response = self.generate_response(user_query)
            
            # Log success
            logger.info(f"[{session_id}] ✅ Query answered in {(datetime.now()-start_time).total_seconds():.2f}s")
            logger.info(f"Query: {user_query[:50]}...")
            logger.info(f"Response confidence: {response['confidence']:.2f}")
            
            self.metrics["successful_responses"] += 1
            
            # Check if should escalate
            if response['confidence'] < 0.5:
                logger.warning(f"[{session_id}] ⚠️ Low confidence - marking for escalation")
                self.metrics["escalations"] += 1
            
            return response
        
        except Exception as e:
            logger.error(f"[{session_id}] ❌ Error: {str(e)}")
            self.metrics["errors"] += 1
            # Return fallback response
            return {"message": "I'm experiencing technical issues. Connecting you to support..."}
    
    def print_metrics(self):
        logger.info(f"Daily Metrics: {self.metrics}")

# Logs go to monitoring system (DataDog, Splunk, etc.)
# Alerts trigger if error_rate > 5% or response_time > 3s
```

---

**4. Identifying When to Escalate:**

```python
def should_escalate(response):
    # Escalate if:
    reasons_to_escalate = []
    
    # Reason 1: Low confidence match
    if response['retrieval_score'] < 0.6:  # Low similarity
        reasons_to_escalate.append("Low match confidence")
    
    # Reason 2: Same issue tried 2+ times
    if response['attempt_count'] >= 2:
        reasons_to_escalate.append("Multiple failed attempts")
    
    # Reason 3: User explicitly asked for human
    if any(phrase in response['user_query'] for phrase in 
           ["talk to human", "agent", "representative", "support team"]):
        reasons_to_escalate.append("User requested human")
    
    # Reason 4: Special keywords (refund, lawsuit, urgent, etc.)
    if any(keyword in response['user_query'].lower() for keyword in
           ["refund", "lawsuit", "urgent", "critical", "emergency"]):
        reasons_to_escalate.append("Sensitive issue detected")
    
    return len(reasons_to_escalate) > 0, reasons_to_escalate

is_escalate, reasons = should_escalate(response)
if is_escalate:
    print(f"Escalating due to: {', '.join(reasons)}")
```

---

**5. Handling 1000+ Concurrent Users:**

```python
# Use async/concurrency + load balancing

import asyncio
from concurrent.futures import ThreadPoolExecutor

class ScalableChatbot:
    def __init__(self, num_workers=10):
        # Thread pool for concurrent requests
        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        # Load-balanced connections to multiple LLM API instances
        self.llm_instances = [OpenAI(api_key=key) for key in llm_keys]
    
    async def handle_user_request(self, user_id, query):
        """Process request asynchronously"""
        # Run in thread pool so it doesn't block
        response = await asyncio.get_event_loop().run_in_executor(
            self.executor,
            self._generate_response,
            user_id,
            query
        )
        return response
    
    def _generate_response(self, user_id, query):
        # Pick LLM instance (round-robin load balancing)
        llm = self.llm_instances[user_id % len(self.llm_instances)]
        # Process and return
        return llm.predict(query)

# Web framework (Flask/FastAPI) receives 1000 requests/sec
# Each gets handled asynchronously without blocking
# 10 worker threads handle actual LLM calls
# Each request gets <3sec response time
```

---

**Architecture Summary:**

```
1000 Users
    ↓
Load Balancer
    ↓
API Server (FastAPI with async handlers)
    ↓
Thread Pool (10 workers) + Cache
    ↓
Vector DB (Chroma - readonly, fast)
    ↓
LLM API (Multiple endpoints for load balancing)
    ↓
Response → Cache → User (<3 seconds)
```

---

## Bonus Challenge Solutions 🌟

### Example 1: Selective Memory Implementation

Selective memory only remembers key facts, not every message.

In [ ]:
# Example: Selective Memory (Educational Implementation)

class SelectiveMemory:
    """Only remember important facts from conversation"""
    
    def __init__(self):
        self.key_facts = []  # Store only important info
    
    def extract_key_facts(self, message):
        """Extract facts from message"""
        # Simple rule-based extraction (in production, use NER or LLM)
        facts = []
        
        keywords = {
            "account": "user",
            "password": "issue_type",
            "billing": "issue_type",
            "urgent": "priority",
            "critical": "priority"
        }
        
        for keyword, fact_type in keywords.items():
            if keyword.lower() in message.lower():
                facts.append({"type": fact_type, "value": keyword})
        
        return facts
    
    def add_message(self, role, message):
        """Add message and extract key facts"""
        facts = self.extract_key_facts(message)
        self.key_facts.extend(facts)
        
        print(f"\n{role.upper()}: {message}")
        if facts:
            print(f"  [Remembered facts: {facts}]")
    
    def get_context(self):
        """Get only the key facts for LLM context"""
        if not self.key_facts:
            return "No key facts yet."
        
        return f"Key facts: {self.key_facts}"

# Test it
memory = SelectiveMemory()
memory.add_message("user", "I have a critical password issue with my account")
memory.add_message("bot", "I understand you have an urgent password problem. Let me help.")
memory.add_message("user", "Yes, I can't log in")
memory.add_message("bot", "I'll help you reset it.")

print(f"\nContext for LLM:\n{memory.get_context()}")

### Example 2: Custom Metadata Filter

In [ ]:
# Example: Custom Metadata Filter (Educational)

class MetadataFilteredRetriever:
    """Retrieve FAQs with custom metadata filtering"""
    
    def __init__(self, vectorstore):
        self.vectorstore = vectorstore
    
    def retrieve_by_region_and_issue(self, query, region, issue_type):
        """Retrieve with multi-field metadata filter"""
        # Filter criteria
        filter_dict = {
            "region": region,
            "issue": issue_type
        }
        
        # Retrieve with filter
        results = self.vectorstore.similarity_search_with_score(
            query=query,
            k=3,
            filter=filter_dict
        )
        
        return results
    
    def smart_fallback_retrieve(self, query, preferred_region=None, preferred_issue=None):
        """Retrieve with fallback if no exact match"""
        # Step 1: Try with both region AND issue
        results = self.retrieve_by_region_and_issue(query, preferred_region, preferred_issue)
        if results:
            print(f"✅ Found match for {preferred_region}/{preferred_issue}")
            return results
        
        # Step 2: Try with just region
        print(f"⚠️ No match for {preferred_region}/{preferred_issue}, trying just region...")
        results = self.vectorstore.similarity_search(
            query,
            k=3,
            filter={"region": preferred_region}
        )
        if results:
            print(f"✅ Found match for {preferred_region}")
            return results
        
        # Step 3: Try with just issue type
        print(f"⚠️ No region match, trying just issue type...")
        results = self.vectorstore.similarity_search(
            query,
            k=3,
            filter={"issue": preferred_issue}
        )
        if results:
            print(f"✅ Found match for {preferred_issue}")
            return results
        
        # Step 4: Return anything relevant
        print(f"⚠️ No metadata match, returning best semantic match...")
        return self.vectorstore.similarity_search(query, k=3)

print("Custom retriever with smart fallback implemented!")
print("Workflow: Region+Issue → Region Only → Issue Only → Best Match")

### Example 3: Simple Chatbot Accuracy Evaluation

In [ ]:
# Example: Evaluation Metric for Chatbot (Educational)

class ChatbotEvaluator:
    """Simple accuracy metric for customer support chatbot"""
    
    def __init__(self):
        self.test_cases = []  # Store test results
    
    def evaluate_response(self, user_query, expected_answer, retrieved_faq):
        """Check if retrieved FAQ matches expected answer"""
        # Simple word overlap metric
        expected_words = set(expected_answer.lower().split())
        retrieved_words = set(retrieved_faq.lower().split())
        
        # Calculate Jaccard similarity
        overlap = expected_words.intersection(retrieved_words)
        union = expected_words.union(retrieved_words)
        similarity = len(overlap) / len(union) if len(union) > 0 else 0
        
        is_correct = similarity > 0.6  # >60% match = correct
        
        self.test_cases.append({
            "query": user_query,
            "similarity": similarity,
            "correct": is_correct
        })
        
        return is_correct, similarity
    
    def calculate_accuracy(self):
        """Calculate overall accuracy percentage"""
        if not self.test_cases:
            return 0.0
        
        correct = sum(1 for tc in self.test_cases if tc["correct"])
        accuracy = (correct / len(self.test_cases)) * 100
        return accuracy
    
    def print_report(self):
        """Print evaluation report"""
        print("\n" + "="*50)
        print("CHATBOT ACCURACY REPORT")
        print("="*50)
        
        for i, tc in enumerate(self.test_cases, 1):
            status = "✅" if tc["correct"] else "❌"
            print(f"{status} Test {i}: {tc['query'][:40]}...")
            print(f"   Similarity: {tc['similarity']:.2%}")
        
        accuracy = self.calculate_accuracy()
        print(f"\nOverall Accuracy: {accuracy:.1f}%")
        print("="*50 + "\n")

# Test it
evaluator = ChatbotEvaluator()

# Test case 1
evaluator.evaluate_response(
    user_query="How do I reset password?",
    expected_answer="Go to Settings > Security and follow the prompts",
    retrieved_faq="For reset password on the web app, open Settings > Security and follow the prompts."
)

# Test case 2
evaluator.evaluate_response(
    user_query="Can I cancel my subscription?",
    expected_answer="Use the cancellation flow under Plans",
    retrieved_faq="For downgrade plan, use the cancellation flow under Plans and give a reason."
)

# Test case 3
evaluator.evaluate_response(
    user_query="What's the weather?",
    expected_answer="Refer to support",
    retrieved_faq="For password reset, open Settings > Security"
)

evaluator.print_report()

---

## Summary of Key Takeaways

✅ **RAG is fundamental** - Retrieval + Generation creates reliable customer support bots

✅ **Semantic search > keyword matching** - Embeddings understand meaning

✅ **Memory management matters** - Choose right strategy for conversation length

✅ **Metadata filtering improves quality** - Region, channel, issue type filters reduce hallucinations

✅ **Prompt engineering is critical** - System prompts control bot behavior

✅ **Production requires thinking ahead** - Caching, monitoring, escalation, concurrency

✅ **Always have a fallback** - When FAQ doesn't have answer, escalate to human

---

Great job working through Week 7! 🎉